# PC-FNO3D en ligne — atom.zelqane.com

Ce notebook sert le modèle **v7** (le modèle retenu) à une **adresse fixe** :

```
navigateur / viz 3D --> atom.zelqane.com (Worker de Rayen, clé injectée côté serveur)
                    --> atom-origin.zelqane.com (tunnel nommé, URL fixe)
                    --> CE notebook : cloudflared --> service sur 127.0.0.1:8000 (GPU)
```

L'adresse ne change pas quand Colab redémarre : il suffit de relancer ce notebook. Tant
qu'il ne tourne pas, la page de Rayen affiche « service endormi ».

**Avant la première exécution — un secret Colab, une seule fois.**
1. Rayen te remet, **de la main à la main** (pas par le salon, pas par git), le contenu des
   identifiants du tunnel `atom` : un texte JSON qui commence par `{"AccountTag"`.
2. Dans Colab, panneau de gauche, icône **clé** (Secrets) → **Ajouter un secret** :
   nom `TUNNEL_CRED_CONTENTS`, valeur = ce texte, et active **Accès au notebook**.

Le secret n'est jamais écrit dans le notebook ni sur le disque : il ne va qu'au processus
cloudflared.

**La clé d'API** : seule son **empreinte** figure ici. Le secret (qui commence par `pcf_`)
t'a été donné par Claude sur ton PC ; c'est toi qui le remets à Rayen, de la main à la
main, pour son Worker. Colab n'en a pas besoin.

**Exécution** : GPU, puis **Exécution → Tout exécuter**. ~5 min : installation, cache des
instants de test, démarrage. Laisse ensuite l'onglet ouvert : le service vit tant que la
session vit.

## 1. Drive et code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import shutil, sys, zipfile
from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")
CIBLE = Path("/content/PFE")


def trouver_archive():
    """L'archive du code sur le Drive : a la racine, puis dans les dossiers probables,
    sous son nom ou renommee par le navigateur (« pcfno3d_code (1).zip »). La plus recente."""
    candidats = []
    for dossier in (MYDRIVE, MYDRIVE / "PFE", MYDRIVE / "Colab Notebooks",
                    MYDRIVE / "aneo_fno3d_runs"):
        if dossier.is_dir():
            candidats += list(dossier.glob("pcfno3d_code*.zip"))
    return max(candidats, key=lambda p: p.stat().st_mtime) if candidats else None


VERSION_MINIMALE = (0, 3, 2)   # 0.3.2 : sante verifiee en local (sinon tout s'arretait)


def version_de(archive) -> tuple:
    """Version du paquet contenu dans l'archive, lue sans l'extraire."""
    import re
    with zipfile.ZipFile(archive) as zf:
        texte = zf.read("src/pcfno3d/__init__.py").decode("utf-8")
    m = re.search(r'__version__ = "(\d+)\.(\d+)\.(\d+)"', texte)
    return tuple(int(x) for x in m.groups()) if m else (0, 0, 0)


ARCHIVE = trouver_archive()
if ARCHIVE is not None and version_de(ARCHIVE) < VERSION_MINIMALE:
    # une vieille copie restee sur le Drive ne doit pas servir en silence
    print(f"{ARCHIVE.name} sur le Drive est trop ancienne (version "
          f"{'.'.join(map(str, version_de(ARCHIVE)))}, il faut au moins "
          f"{'.'.join(map(str, VERSION_MINIMALE))}) : choisis la nouvelle sur ton PC.")
    ARCHIVE = None
if ARCHIVE is None:
    # absente du Drive, ou perimee : on la televerse depuis le PC, puis on la garde
    print("Clique sur « Choisir des fichiers » et prends l'archive sur ton PC :")
    print(r"C:\Users\inesr\OneDrive\Desktop\PFE\artifacts\pcfno3d_code.zip")
    from google.colab import files
    recu = files.upload()
    nom = next((n for n in recu if n.startswith("pcfno3d_code") and n.endswith(".zip")), None)
    if nom is None:
        raise FileNotFoundError(f"fichier recu : {list(recu)} — il faut pcfno3d_code.zip")
    ARCHIVE = MYDRIVE / "pcfno3d_code.zip"
    ARCHIVE.write_bytes(recu[nom])
    if version_de(ARCHIVE) < VERSION_MINIMALE:
        raise RuntimeError("l'archive choisie est elle aussi trop ancienne. Sur ton PC : "
                           "python scripts/package_for_colab.py, puis choisis le fichier produit.")
    print(f"archive gardee sur le Drive pour les prochaines sessions : {ARCHIVE}")
print(f"archive : {ARCHIVE} (version {'.'.join(map(str, version_de(ARCHIVE)))})")

if CIBLE.exists():
    shutil.rmtree(CIBLE)
CIBLE.mkdir(parents=True)
with zipfile.ZipFile(ARCHIVE) as zf:
    zf.extractall(CIBLE)

# tampon ecrit par package_for_colab.py : le commit exact du code qui va tourner
tampon = CIBLE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
# --no-deps : Colab a deja numpy, pandas, scipy, pyarrow, torch, pyyaml et matplotlib. Laisser
# pip « resoudre » les dependances l'avait fait redescendre numpy de 2.x a 1.26, ce qui
# casse pandas et scipy, compiles pour numpy 2.
!pip install -q --no-deps -e . 2>&1 | tail -2
# le service web : aucune de ces bibliotheques ne touche a numpy
!pip install -q fastapi uvicorn python-multipart httpx mcp==2.3.0 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]
import numpy
if int(numpy.__version__.split(".")[0]) < 2:
    raise RuntimeError(
        f"numpy {numpy.__version__} : une ancienne installation l'a fait redescendre, et pandas "
        "et scipy (compiles pour numpy 2) risquent de planter. Execution -> Deconnecter et "
        "supprimer l'environnement d'execution, puis Tout executer.")


def lancer(commande: str) -> None:
    """Exécute une commande en affichant sa sortie, et ARRÊTE le notebook si elle échoue.

    Une ligne `!commande` qui échoue ne stoppe pas « Tout exécuter » : les cellules
    suivantes partiraient d'un résultat absent. D'où ce garde-fou.
    """
    ip = get_ipython()
    ip.system(commande)
    code = ip.user_ns.get("_exit_code", 0)
    if code != 0:
        raise RuntimeError(f"échec (code {code}) de : {commande}\n"
                           "Le notebook s'arrête ici : lis l'erreur juste au-dessus.")


print("prêt")

## 2. Instants de test

Le service lit des instants mis en cache. On ne construit que les **15 jours de test**
(166 instants, ceux sur lesquels les DSO sont mesurés) : quelques minutes au lieu de 20.
Si le cache complet est déjà sur le disque de Colab (même session que le notebook v8), rien
n'est refait.

In [ ]:
import json
import pandas as pd

DATASET = MYDRIVE / "aneo_dataset_3d_v2"
CACHE = Path("/content/aneo_cache_v7")
jours_test = json.loads((DATASET / "split_days.json").read_text())["test_days"]
deja = set(pd.read_csv(CACHE / "index.csv")["day"]) if (CACHE / "index.csv").exists() else set()
manque = [d for d in jours_test if d not in deja]
if manque:
    print(f"construction du cache : {len(manque)} jour(s) de test")
    lancer(f"python -m pcfno3d -c colab build-cache --days {','.join(map(str, manque))}")
index = pd.read_csv(CACHE / "index.csv")
print(f"cache prêt : {int((index['split'] == 'test').sum())} instants de test")

## 3. Volumes 3D pour la visualisation

La vue 3D de Rayen lit le volume complet de chaque instant (`/api/export/<instant>/...`,
Float32). On les produit ici une fois par session, sur le disque de Colab (~1 Go, ~1 à
2 min sur GPU) : un pas sur deux à l'horizontale (125 × 250 × 12), ~1,5 Mo par variable.
Les instants déjà exportés sont sautés.

In [ ]:
lancer("python -m pcfno3d -c colab export-web --split test --stride 2")

## 4. Démarrer le service derrière le tunnel nommé

Ce qui est public ici : l'identifiant du tunnel, le nom d'hôte et l'**empreinte** de la clé.
Rien de cela ne permet d'entrer. Le seul secret, `TUNNEL_CRED_CONTENTS`, est lu dans les
secrets Colab.

In [ ]:
from google.colab import userdata
from service.colab import serve_public

TUNNEL_ID = "e552f718-2478-4fe3-82b4-8d33a4b9eb6e"
HOTE = "atom-origin.zelqane.com"
CLE_NOM = "atom-worker"                      # la clé de lecture du Worker de Rayen
CLE_EMPREINTE = "ce7965c54284eb6d1a274119839a71d9785731609d04b17bd08fb6da7d2c30fd"

try:
    identifiants = userdata.get("TUNNEL_CRED_CONTENTS")
except Exception as exc:                     # secret absent, ou accès non accordé
    raise RuntimeError(
        "secret Colab TUNNEL_CRED_CONTENTS introuvable ou non autorisé : panneau de gauche, "
        "icône clé, ajoute-le (contenu remis par Rayen) et active « Accès au notebook ».") from None

info = serve_public(config="colab", key_name=CLE_NOM, scope="read", key_hash=CLE_EMPREINTE,
                    named_tunnel={"id": TUNNEL_ID, "hostname": HOTE,
                                  "credentials": identifiants})
del identifiants

## 5. Vérifier de l'extérieur

In [ ]:
import urllib.request

# User-Agent explicite : Cloudflare refuse souvent celui de urllib par defaut
req = urllib.request.Request(f"https://{HOTE}/api/health",
                             headers={"User-Agent": "pcfno3d-colab/1.0"})
with urllib.request.urlopen(req, timeout=60) as r:
    sante = json.loads(r.read().decode())
print(json.dumps(sante, ensure_ascii=False, indent=1))
assert sante["status"] == "ok", "le service ne répond pas correctement par le tunnel"
print(f"\nEN LIGNE : https://{HOTE}  — préviens Rayen (sans coller de clé).")

## 6. Garder en vie

Cette cellule tourne indéfiniment et affiche l'état toutes les 10 minutes. Arrête-la
(carré) quand tu veux couper : la cellule suivante arrête proprement le service.

In [ ]:
import time
try:
    while True:
        tunnel_ok = info["tunnel"].alive
        service_ok = info["process"].poll() is None
        print(time.strftime("%H:%M"), "| tunnel", "ok" if tunnel_ok else "ARRÊTÉ",
              "| service", "ok" if service_ok else "ARRÊTÉ", flush=True)
        if not (tunnel_ok and service_ok):
            print("journaux :", info["log_path"], "et", info["tunnel"].log_path)
            break
        time.sleep(600)
except KeyboardInterrupt:
    print("surveillance arrêtée — le service tourne toujours")

In [ ]:
from service.colab import stop
stop(info)
print("service et tunnel arrêtés : atom.zelqane.com repasse en « endormi »")